# 05-2. Supervisor 패턴 — Jev(System One Model) 기반 동적 라우팅
**학습 목표:** 05-1번과 같은 Supervisor 구조에서, 라우팅 판단만 일반 LLM 대신 TypeSafe AI의 Jev(System One Model)로 바꾼다. Jev는 텍스트를 생성하지 않고 미리 정의한 선택지 중 하나와 확률을 구조화된 형태로 반환하는 **판단 전용 모델**이다. 이 Notebook은 `OPENAI_API_KEY`(답변 생성)와 `TYPESAFE_API_KEY`(라우팅 판단) 둘 다 필요하며, 실행마다 두 서비스 모두 비용이 발생한다.


## 1. 핵심 개념 — 05-1과 무엇이 다른가
05-1번에서는 Supervisor가 LLM에게 "라벨만 답하라"고 프롬프트로 지시했다. 모델은 여전히 텍스트를 생성한 뒤 그 텍스트가 우연히 유효한 라벨이길 기대하는 구조다. Jev는 애초에 텍스트를 생성하지 않고, 개발자가 정의한 선택지(`Choice`) 중 하나를 확률(probabilities)과 함께 반환하도록 설계된 모델이라 이 불확실성이 구조적으로 줄어든다. 속도도 70~500ms로 빠르고, 비용도 입력 토큰 100만 개당 \$0.042로 저렴하다(2026-09-15 TypeSafe AI 발표 기준).

```mermaid
flowchart TD
  S[START] --> SV[Supervisor: Jev가 라벨+확률 판단]
  SV -->|fit_agent| F[Fit Agent]
  SV -->|interview_agent| I[Interview Agent]
  SV -->|strategy_agent| T[Strategy Agent]
  F --> E[END]
  I --> E
  T --> E
```

Graph 구조(`add_conditional_edges`로 세 전문 Agent 중 하나로 보내는 것)는 05-1번과 완전히 같다. 바뀌는 부분은 Supervisor Node 하나뿐이다.


## 2. 사전 준비
`pip install typesafe-sdk`로 SDK를 설치한다. `.env`에 `TYPESAFE_API_KEY=ts_...`를 추가한다 (`.env.example`에 자리만 마련해 두었다). `common.ai_model`을 import하는 시점에 `common.config`가 `.env`를 이미 로드하므로, `TypeSafeClient()`는 별도 설정 없이 환경 변수에서 키를 읽는다.


In [ ]:
from typing import TypedDict

from langgraph.graph import START, END, StateGraph
from typesafe_sdk import Choice, TypeSafeClient

from common.ai_model import get_llm_model

llm = get_llm_model()

print(llm.model_name)


## 3. 샘플 데이터
05-1번과 동일한 채용공고·이력서를 그대로 사용한다.


In [ ]:
job_posting = {
    "title": "백엔드 개발자",
    "required_skills": ["Python", "SQL", "REST API"],
    "preferred_skills": ["LangGraph", "Docker"],
    "required_experience": "실무 2년 이상",
}

candidate_resume = {
    "name": "홍길동",
    "skills": ["Python", "LangGraph", "Docker"],
    "experience": "스타트업에서 1년간 백엔드 인턴 경험",
}

print(job_posting["title"], "/", candidate_resume["name"])


## 4. State 설계
`route_confidence`가 05-1번과 다른 점이다. Jev는 선택한 라벨뿐 아니라 각 선택지의 확률도 함께 주므로, 그 근거를 State에 남겨 Review 단계나 로그에서 "왜 이렇게 라우팅됐는지" 추적할 수 있다.


In [ ]:
class SupportState(TypedDict, total=False):
    user_query: str
    job_posting: dict
    candidate_resume: dict
    route: str
    route_confidence: dict
    agent_response: str


ROUTES = ("fit_agent", "interview_agent", "strategy_agent")


## 5. Supervisor — Jev에게 라우팅만 맡긴다
`Choice`의 `criteria`에 선택지별 설명을 주면, Jev는 그 설명과 `state`(판단 대상 텍스트)를 비교해 가장 적합한 선택지 하나와 선택지별 확률 분포를 반환한다. 05-1번의 Supervisor와 달리 자유 텍스트 생성이 아예 없으므로, "모델이 라벨 아닌 다른 말을 했을 때의 기본값 처리"가 필요 없다.


In [ ]:
ROUTE_CRITERIA = {
    "fit_agent": "지원자 역량·자격이 채용공고 요건에 맞는지에 대한 질문",
    "interview_agent": "면접 준비, 예상 질문에 대한 질문",
    "strategy_agent": "지원 전략, 취업 방향에 대한 질문",
}


def supervisor(state: SupportState) -> dict:
    """LLM 대신 Jev(System One Model)에게 라우팅 판단만 맡긴다."""
    # state 인자는 Jev가 판단할 대상 텍스트다(이 Graph의 SupportState와 이름만 같은 별개 개념).
    with TypeSafeClient() as client:
        response = client.system_one(
            model="jev-1.13.0",
            state={"user_query": state["user_query"]},
            questions={
                "route": Choice(
                    instructions="이 질문은 세 전문 Agent 중 누구에게 가야 하는가?",
                    criteria=ROUTE_CRITERIA,
                ),
            },
        )
    answer = response.choices["route"]
    # 선택된 라벨뿐 아니라 선택지별 확률도 같이 남겨, 애매한 판단(확률이 고르게 분산된 경우)을 나중에 들여다볼 수 있게 한다.
    return {"route": answer.choice, "route_confidence": answer.probabilities}


def route_after_supervisor(state: SupportState) -> str:
    """add_conditional_edges가 호출하는 판단 함수. supervisor가 쓴 State Key를 그대로 읽어 다음 Node 이름을 고른다."""
    return state["route"]


## 6. 전문 Agent 세 개
05-1번과 완전히 동일하다. Jev는 라우팅 판단만 담당하고, 실제 답변 생성은 여전히 일반 LLM이 한다.


In [ ]:
def fit_agent(state: SupportState) -> dict:
    """역량·자격 비교 질문에 답한다."""
    prompt = (
        "당신은 역량 비교 담당자다. 채용공고 요건과 지원자 보유 역량을 비교해 질문에 답하라. "
        "공고·이력서에 없는 내용은 추측하지 마라.\n"
        f"채용공고: {state['job_posting']}\n이력서: {state['candidate_resume']}\n"
        f"질문: {state['user_query']}"
    )
    return {"agent_response": llm.invoke(prompt).content}


def interview_agent(state: SupportState) -> dict:
    """면접 준비 질문에 답한다."""
    prompt = (
        "당신은 면접 코치다. 채용공고와 이력서를 참고해 예상 질문과 답변 방향을 제시하라.\n"
        f"채용공고: {state['job_posting']}\n이력서: {state['candidate_resume']}\n"
        f"질문: {state['user_query']}"
    )
    return {"agent_response": llm.invoke(prompt).content}


def strategy_agent(state: SupportState) -> dict:
    """지원 전략 질문에 답한다."""
    prompt = (
        "당신은 취업 전략 어드바이저다. 지원자의 강점과 공고 요건을 고려해 지원 전략을 제시하라.\n"
        f"채용공고: {state['job_posting']}\n이력서: {state['candidate_resume']}\n"
        f"질문: {state['user_query']}"
    )
    return {"agent_response": llm.invoke(prompt).content}


## 7. Graph 구성 — `add_conditional_edges`
05-1번과 코드가 완전히 동일하다. Graph 구조는 "판단을 누가 하는가"와 무관하다는 것을 보여 준다.


In [ ]:
builder = StateGraph(SupportState)
builder.add_node("supervisor", supervisor)
builder.add_node("fit_agent", fit_agent)
builder.add_node("interview_agent", interview_agent)
builder.add_node("strategy_agent", strategy_agent)

builder.add_edge(START, "supervisor")
builder.add_conditional_edges("supervisor", route_after_supervisor, {r: r for r in ROUTES})
builder.add_edge("fit_agent", END)
builder.add_edge("interview_agent", END)
builder.add_edge("strategy_agent", END)
graph = builder.compile()


## 8. Graph 시각화


In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    # 시각화 서비스가 없어도 Graph의 실행은 계속된다.
    print("PNG 시각화 불가:", type(exc).__name__)
    print(graph.get_graph().draw_mermaid())


## 9. 실행과 라우팅 검증
05-1번과 같은 질문 세 개로 비교한다. 이번에는 `route_confidence`도 함께 출력해, Jev가 라벨을 얼마나 확신했는지 본다. 질문마다 Jev 호출 1회 + 전문 Agent의 LLM 호출 1회, 총 두 서비스에 비용이 발생한다.


In [ ]:
test_cases = [
    ("SQL 역량이 부족한데 이 공고에 지원할 수 있을까요?", "fit_agent"),
    ("이 공고 면접에서 어떤 질문이 나올까요?", "interview_agent"),
    ("이 공고에 지원할 때 어떤 전략으로 접근해야 하나요?", "strategy_agent"),
]

results = []
for query, expected_route in test_cases:
    state = {"user_query": query, "job_posting": job_posting, "candidate_resume": candidate_resume}
    result = graph.invoke(state)
    results.append(result)
    print(f"질문: {query}")
    print(f"라우팅: {result['route']} (기대: {expected_route}) | 확률: {result['route_confidence']}")
    print(result["agent_response"][:200], "...\n")
    assert result["route"] == expected_route


## 10. LLM 라우팅과 Jev 라우팅 비교, 그리고 수강생 미션
| 방식 | 지연 | 비용 | 출력 형태 | 신뢰성 |
|---|---|---|---|---|
| 05-1: LLM 라우팅 | 초 단위 | 전체 모델 단가 | 자유 텍스트(라벨이길 기대) | 프롬프트 품질에 좌우 |
| 05-2: Jev 라우팅 | 70~500ms | \$0.042/백만 입력 토큰 | 구조화된 선택지 + 확률 | 선택지 밖 답이 나올 수 없음 |

Jev는 "판단"에 특화된 만큼 자유 형식 응답 생성에는 쓸 수 없고, 외부 유료 API·네트워크 의존성이 추가된다는 트레이드오프가 있다. 그래서 "라우팅·분류는 Jev, 실제 분석·작성은 기존 LLM"으로 역할을 나누는 조합이 일반적이다.

**미션:** `route_confidence`에서 선택된 라벨의 확률이 0.6 미만이면, Jev의 판단을 그대로 쓰지 않고 05-1번의 LLM 기반 `supervisor`로 한 번 더 확인하는 하이브리드 Supervisor를 만들어라.
